# kNN relevance experiment

An alternative to the query-based AI check in `crawl_corpus.ipynb`:

1. take **X related** seed Articles (directly involved in US–Japan engagement) and **Y unrelated** ones
   (close look-alikes + clearly unrelated) from `config/knn_seeds.yaml`;
2. use their e5 vectors (title + war-related sentences, already in `cache.sqlite`);
3. classify every other Article by its **K nearest seeds** (similarity-weighted vote).

Runs **offline**: everything comes from `Data/interim/` written by `crawl_corpus.ipynb`.
Evaluation uses the **holdout** labels in `gray_labels.yaml`, which are never seeds.
K is chosen by leave-one-out on the seeds, not on the holdout.

In [1]:
import collections
import json
import random
import sqlite3
from pathlib import Path

import numpy as np
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "Processing" else Path.cwd()
CONFIG_DIR = ROOT / "Processing" / "config"
INTERIM = ROOT / "Data" / "interim"
RESULTS_PATH = INTERIM / "knn_results.json"
PREDICTIONS_PATH = INTERIM / "knn_gray_zone.csv"

E5_VARIANT = "intfloat/e5-base-v2|war-sentences-v1"  # must match crawl_corpus.ipynb
K_GRID = [1, 3, 5, 7, 9, 11, 15, 21, 31]

seeds_cfg = yaml.safe_load((CONFIG_DIR / "knn_seeds.yaml").read_text())
gray_labels = yaml.safe_load((CONFIG_DIR / "gray_labels.yaml").read_text())
gold = yaml.safe_load((CONFIG_DIR / "gold_set.yaml").read_text())
gold_titles = [t for titles in gold.values() for t in titles]

db = sqlite3.connect(f"file:{INTERIM / 'cache.sqlite'}?mode=ro", uri=True)
vectors = {t: np.frombuffer(v, dtype=np.float32)
           for t, v in db.execute("SELECT title, vec FROM vectors WHERE variant=?", (E5_VARIANT,))}
rows = {}
for name in ("tagged.jsonl", "excluded.jsonl", "untagged.jsonl"):
    with (INTERIM / name).open() as f:
        for line in f:
            r = json.loads(line)
            rows[r["title"]] = r
print(f"{len(vectors):,} cached vectors, {len(rows):,} classified Articles")

14,590 cached vectors, 14,591 classified Articles


## 1. Seeds

In [2]:
holdout = {t: bool(y) for t, y in gray_labels["holdout"].items()}
calibration = {t: bool(y) for t, y in gray_labels["calibration"].items()}

related = list(seeds_cfg["related"].get("extra") or [])
if seeds_cfg["related"].get("include_gold_set"):
    related += gold_titles
if seeds_cfg["related"].get("include_calibration_labels"):
    related += [t for t, y in calibration.items() if y]

close = list(seeds_cfg["unrelated"].get("close") or [])
if seeds_cfg["unrelated"].get("include_calibration_labels"):
    close += [t for t, y in calibration.items() if not y]

far_pool = sorted(t for t, r in rows.items() if r["reason"] in ("not tied to any in-scope battle", "media / culture")
                  and t in vectors and t not in holdout and t not in close)
far = random.Random(seeds_cfg["unrelated"].get("far_seed", 0)).sample(far_pool, seeds_cfg["unrelated"].get("far_sample", 100))


def clean(titles, label):
    kept = list(dict.fromkeys(t for t in titles if t in vectors and t not in holdout))
    missing = [t for t in titles if t not in vectors]
    if missing:
        print(f"  {label}: {len(missing)} not in cache, skipped: {missing[:5]}")
    return kept


related, close, far = clean(related, "related"), clean(close, "close"), clean(far, "far")
overlap = set(related) & (set(close) | set(far))
assert not overlap, f"seed in both classes: {overlap}"
print(f"X related = {len(related)} · Y unrelated = {len(close) + len(far)} ({len(close)} close, {len(far)} far)")

X related = 87 · Y unrelated = 182 (82 close, 100 far)


## 2. kNN classifier
Score = similarity-weighted share of related seeds among the K nearest. Related if score ≥ 0.5.

In [3]:
def build(seed_related, seed_unrelated):
    titles = seed_related + seed_unrelated
    return np.stack([vectors[t] for t in titles]), np.array([1] * len(seed_related) + [0] * len(seed_unrelated)), titles


def knn_scores(queries, S, y, k, exclude_self=None, weighted=True):
    sims = np.stack([vectors[q] for q in queries]) @ S.T
    if exclude_self is not None:
        for i, j in exclude_self:
            sims[i, j] = -np.inf
    top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
    s = np.take_along_axis(sims, top, axis=1)
    w = np.clip(s, 0, None) if weighted else np.ones_like(s)
    return (w * y[top]).sum(1) / np.maximum(w.sum(1), 1e-9)


def metrics(scores, labels, cut=0.5):
    labels = np.asarray(labels, dtype=bool)
    pred = np.asarray(scores) >= cut
    tp, fp = int((pred & labels).sum()), int((pred & ~labels).sum())
    fn, tn = int((~pred & labels).sum()), int((~pred & ~labels).sum())
    order = np.argsort(scores); ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores) + 1)
    npos, nneg = labels.sum(), (~labels).sum()
    auc = (ranks[labels].sum() - npos * (npos + 1) / 2) / (npos * nneg) if npos and nneg else float("nan")
    return {"accuracy": (tp + tn) / len(labels), "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),
            "auc": float(auc), "tp": tp, "fp": fp, "fn": fn, "tn": tn}


S, y, seed_titles = build(related, close + far)

## 3. Choose K by leave-one-out on the seeds

In [4]:
loo = {}
for weighted in (True, False):
    for k in K_GRID:
        sc = knn_scores(seed_titles, S, y, k, exclude_self=[(i, i) for i in range(len(seed_titles))], weighted=weighted)
        loo[(k, weighted)] = metrics(sc, y)
best_k, best_w = max(loo, key=lambda kw: (round(loo[kw]["accuracy"], 3), loo[kw]["auc"]))
print(" K  vote      LOO acc  AUC")
for (k, w), m in loo.items():
    print(f"{k:2}  {'weighted' if w else 'majority'}  {m['accuracy']:.0%}   {m['auc']:.2f}{'  ← chosen' if (k, w) == (best_k, best_w) else ''}")

 K  vote      LOO acc  AUC
 1  weighted  73%   0.79
 3  weighted  77%   0.85
 5  weighted  77%   0.87
 7  weighted  76%   0.86
 9  weighted  75%   0.87
11  weighted  77%   0.88
15  weighted  78%   0.89
21  weighted  77%   0.88
31  weighted  80%   0.88  ← chosen
 1  majority  73%   0.79
 3  majority  77%   0.85
 5  majority  77%   0.87
 7  majority  76%   0.86
 9  majority  75%   0.87
11  majority  77%   0.88
15  majority  78%   0.90
21  majority  77%   0.87
31  majority  80%   0.88


## 4. Holdout evaluation (68 labels never used as seeds)
Compared with the current AI check (query contrast with calibrated thresholds) on the same Articles.

In [5]:
hold_titles = [t for t in holdout if t in vectors]
hold_y = [holdout[t] for t in hold_titles]
holdout_table = []
for weighted in (True, False):
    for k in K_GRID:
        m = metrics(knn_scores(hold_titles, S, y, k, weighted=weighted), hold_y)
        holdout_table.append({"k": k, "weighted": weighted, **m})
chosen = metrics(knn_scores(hold_titles, S, y, best_k, weighted=best_w), hold_y)

rel = np.array([rows[t]["relevance"] for t in hold_titles])
cal_pts = [(rows[t]["relevance"], v) for t, v in calibration.items() if t in rows and rows[t]["relevance"] is not None]
cal_s, cal_y = np.array([p for p, _ in cal_pts]), np.array([v for _, v in cal_pts])
single_cut = max(np.unique(cal_s), key=lambda c: ((cal_s >= c) == cal_y).mean())
baseline = metrics(rel, hold_y, cut=single_cut)
current = collections.Counter(rows[t]["decided_by"] for t in hold_titles)

print(f"kNN (K={best_k}, {'weighted' if best_w else 'majority'}): accuracy {chosen['accuracy']:.0%} · precision "
      f"{chosen['precision']:.0%} · recall {chosen['recall']:.0%} · AUC {chosen['auc']:.2f}")
print(f"current AI check, one cut: accuracy {baseline['accuracy']:.0%} · precision {baseline['precision']:.0%} · "
      f"recall {baseline['recall']:.0%} · AUC {baseline['auc']:.2f}")
print("\n K  vote      acc   prec  recall AUC")
for r in holdout_table:
    print(f"{r['k']:2}  {'weighted' if r['weighted'] else 'majority'}  {r['accuracy']:.0%}  {r['precision']:.0%}  {r['recall']:.0%}   {r['auc']:.2f}")

kNN (K=31, weighted): accuracy 74% · precision 79% · recall 52% · AUC 0.86
current AI check, one cut: accuracy 88% · precision 96% · recall 76% · AUC 0.96

 K  vote      acc   prec  recall AUC
 1  weighted  75%  77%  59%   0.69
 3  weighted  78%  82%  62%   0.82
 5  weighted  74%  74%  59%   0.83
 7  weighted  72%  73%  55%   0.85
 9  weighted  76%  78%  62%   0.86
11  weighted  74%  79%  52%   0.88
15  weighted  76%  81%  59%   0.87
21  weighted  74%  79%  52%   0.87
31  weighted  74%  79%  52%   0.86
 1  majority  75%  77%  59%   0.69
 3  majority  78%  82%  62%   0.80
 5  majority  74%  74%  59%   0.81
 7  majority  72%  73%  55%   0.85
 9  majority  76%  78%  62%   0.86
11  majority  74%  79%  52%   0.88
15  majority  76%  81%  59%   0.87
21  majority  74%  79%  52%   0.86
31  majority  74%  79%  52%   0.86


## 5. Ablations – which seeds matter?

In [6]:
def holdout_acc(rel_seeds, unrel_seeds, k=best_k, w=best_w):
    S_, y_, _ = build(rel_seeds, unrel_seeds)
    return metrics(knn_scores(hold_titles, S_, y_, min(k, len(y_)), weighted=w), hold_y)


ablations = {
    "all seeds": holdout_acc(related, close + far),
    "no far negatives": holdout_acc(related, close),
    "no close negatives": holdout_acc(related, far),
    "no hand-picked close negatives": holdout_acc(related, [t for t in close if t not in (seeds_cfg["unrelated"].get("close") or [])] + far),
    "related = gold set only": holdout_acc([t for t in related if t in gold_titles], close + far),
}
rng = random.Random(1)
for frac in (0.25, 0.5, 0.75):
    accs = [holdout_acc(rng.sample(related, max(3, int(len(related) * frac))),
                        rng.sample(close + far, max(3, int(len(close + far) * frac))))["accuracy"] for _ in range(10)]
    ablations[f"{int(frac * 100)}% of seeds (mean of 10)"] = {"accuracy": float(np.mean(accs)), "auc": float("nan"),
                                                          "precision": float("nan"), "recall": float("nan")}
for name, m in ablations.items():
    print(f"{name:34} acc {m['accuracy']:.0%}  AUC {m['auc']:.2f}")

all seeds                          acc 74%  AUC 0.86
no far negatives                   acc 85%  AUC 0.93
no close negatives                 acc 38%  AUC 0.41
no hand-picked close negatives     acc 72%  AUC 0.78
related = gold set only            acc 59%  AUC 0.75
25% of seeds (mean of 10)          acc 71%  AUC nan
50% of seeds (mean of 10)          acc 70%  AUC nan
75% of seeds (mean of 10)          acc 70%  AUC nan


## 6. Apply to the whole Gray zone and compare with the current AI check

In [7]:
gray = [t for t, r in rows.items() if r["rule_verdict"] == "gray" and t in vectors and t not in set(seed_titles)]
gray_scores = knn_scores(gray, S, y, best_k, weighted=best_w)
current_dec = {t: ("accept" if rows[t]["in_scope"] else "review" if rows[t]["decided_by"] == "review" else "reject")
               for t in gray}
agree = collections.Counter((current_dec[t], "related" if s >= 0.5 else "unrelated") for t, s in zip(gray, gray_scores))
print(f"Gray zone: {len(gray):,} Articles · kNN says related {int((gray_scores >= 0.5).sum()):,}, unrelated {int((gray_scores < 0.5).sum()):,}")
for (cur, knn), n in sorted(agree.items()):
    print(f"  current {cur:7} · kNN {knn:9}: {n:,}")

# sanity: how kNN rates Articles the rules already decided
for verdict in ("in", "out"):
    ts = [t for t, r in rows.items() if r["rule_verdict"] == verdict and t in vectors and t not in set(seed_titles)]
    sc = knn_scores(ts, S, y, best_k, weighted=best_w)
    print(f"rules said {verdict:3}: kNN agrees on {((sc >= 0.5) == (verdict == 'in')).mean():.0%} of {len(ts):,}")

with PREDICTIONS_PATH.open("w") as f:
    f.write("title,primary_tag,knn_score,knn_says,current_decision,current_relevance\n")
    for t, s in sorted(zip(gray, gray_scores), key=lambda x: -x[1]):
        r = rows[t]
        f.write(f"\"{t}\",{r['primary_tag'] or ''},{s:.3f},{'related' if s >= 0.5 else 'unrelated'},{current_dec[t]},{r['relevance']}\n")
print(f"wrote {PREDICTIONS_PATH}")

Gray zone: 2,975 Articles · kNN says related 1,158, unrelated 1,817
  current accept  · kNN related  : 460
  current accept  · kNN unrelated: 195
  current reject  · kNN related  : 124
  current reject  · kNN unrelated: 1,011
  current review  · kNN related  : 574
  current review  · kNN unrelated: 611
rules said in : kNN agrees on 71% of 4,322
rules said out: kNN agrees on 76% of 7,024


wrote /Users/nivgreenstein/Desktop/projects/geoai/Corpus2GeoRAG_Pacific/Data/interim/knn_gray_zone.csv


## 7. Errors on the holdout, and the known hard cases

In [8]:
hold_sc = knn_scores(hold_titles, S, y, best_k, weighted=best_w)
errors = [(t, l, float(s)) for t, l, s in zip(hold_titles, hold_y, hold_sc) if (s >= 0.5) != l]
print("holdout errors:")
for t, l, s in errors:
    print(f"  {'missed (should be in)' if l else 'false alarm (should be out)':28} {s:.2f}  {t}")
known = {"Yoshijirō Umezu": True, "4th Marine Aircraft Wing": True, "Japanese destroyer Yanagi (1944)": True,
         "USS Bache (DD-470)": True, "William E. Dyess": True, "Hong Xuezhi": False}
known = {t: v for t, v in known.items() if t in vectors and t not in set(seed_titles)}
ks = knn_scores(list(known), S, y, best_k, weighted=best_w)
print("known hard cases:")
for (t, l), s in zip(known.items(), ks):
    print(f"  {'✓' if (s >= 0.5) == l else '✗'} {t:36} should be {'in ' if l else 'out'} kNN {s:.2f}")

holdout errors:
  false alarm (should be out)  0.52  Morningside Air Raid Shelter
  false alarm (should be out)  0.61  British Commonwealth Air Training Plan
  missed (should be in)        0.26  Marshall Parker
  missed (should be in)        0.42  Type 99 cannon
  missed (should be in)        0.39  Ayatosan Maru
  false alarm (should be out)  0.71  Pacific Proving Grounds
  missed (should be in)        0.39  Albert W. Waldron
  missed (should be in)        0.26  Jack Sanders (American football)
  missed (should be in)        0.42  USS Marion County
  false alarm (should be out)  0.55  Almirante Latorre-class battleship
  missed (should be in)        0.42  Geoanna
  missed (should be in)        0.26  George W. D'Artois
  missed (should be in)        0.39  Leyburn Airfield
  missed (should be in)        0.45  USS LST-179
  missed (should be in)        0.19  Bill Greason
  missed (should be in)        0.45  USS Lamar (APA-47)
  missed (should be in)        0.39  Ruperto Kangleon
  missed 

## 8. Improvements: balanced voting, look-alike negatives only, ensemble
The ablations show the far negatives *hurt*: with 182 unrelated vs 87 related seeds, votes lean "unrelated"
(low recall). Balanced voting divides each side's votes by its seed count. K is again chosen by
leave-one-out on the seeds. The ensemble averages the ranks of the kNN score and the current query score.

In [9]:
def knn_balanced(queries, S, y, k, exclude_self=None):
    sims = np.stack([vectors[q] for q in queries]) @ S.T
    if exclude_self is not None:
        for i, j in exclude_self:
            sims[i, j] = -np.inf
    top = np.argpartition(-sims, k - 1, axis=1)[:, :k]
    w = np.clip(np.take_along_axis(sims, top, axis=1), 0, None)
    pos = (w * (y[top] == 1)).sum(1) / (y == 1).sum()
    neg = (w * (y[top] == 0)).sum(1) / (y == 0).sum()
    return pos / np.maximum(pos + neg, 1e-9)


def rank01(x):
    return np.argsort(np.argsort(x)) / (len(x) - 1)


variants = {}
for name, unrel in (("balanced, all negatives", close + far), ("balanced, look-alike negatives only", close)):
    S_, y_, t_ = build(related, unrel)
    loo_acc = {k: metrics(knn_balanced(t_, S_, y_, k, exclude_self=[(i, i) for i in range(len(t_))]), y_)["accuracy"]
               for k in K_GRID}
    k_ = max(loo_acc, key=lambda k: (round(loo_acc[k], 3), -k))
    variants[name] = {"k": k_, "loo_accuracy": loo_acc[k_], **metrics(knn_balanced(hold_titles, S_, y_, k_), hold_y)}
S_best, y_best, _ = build(related, close)
k_best = variants["balanced, look-alike negatives only"]["k"]
ens = 0.5 * rank01(knn_balanced(hold_titles, S_best, y_best, k_best)) + 0.5 * rank01(rel)
variants["ensemble: kNN + current query (rank average)"] = {"k": k_best, **metrics(ens, hold_y, cut=np.inf)}
variants["current query method, one cut (baseline)"] = {"k": None, **baseline}
print("variant                                         K   acc   prec  recall AUC")
for name, m in variants.items():
    acc = "  -  " if name.startswith("ensemble") else f"{m['accuracy']:.0%} "
    prec = "  -  " if name.startswith("ensemble") else f"{m['precision']:.0%} "
    recall = "  -  " if name.startswith("ensemble") else f"{m['recall']:.0%} "
    print(f"{name:46} {str(m['k'] or '-'):>3}  {acc} {prec} {recall}  {m['auc']:.2f}")

bal_gray = knn_balanced(gray, S_best, y_best, k_best)
agree_bal = collections.Counter((current_dec[t], "related" if s >= 0.5 else "unrelated") for t, s in zip(gray, bal_gray))
print(f"\nGray zone with the best kNN: related {int((bal_gray >= 0.5).sum()):,} / unrelated {int((bal_gray < 0.5).sum()):,}")
for (cur, knn), n in sorted(agree_bal.items()):
    print(f"  current {cur:7} · kNN {knn:9}: {n:,}")

variant                                         K   acc   prec  recall AUC
balanced, all negatives                          1  75%  77%  59%   0.69
balanced, look-alike negatives only              7  84%  75%  93%   0.92
ensemble: kNN + current query (rank average)     7    -     -     -    0.96
current query method, one cut (baseline)         -  88%  96%  76%   0.96

Gray zone with the best kNN: related 1,901 / unrelated 1,074
  current accept  · kNN related  : 649
  current accept  · kNN unrelated: 6
  current reject  · kNN related  : 296
  current reject  · kNN unrelated: 839
  current review  · kNN related  : 956
  current review  · kNN unrelated: 229


In [10]:
RESULTS_PATH.write_text(json.dumps({
    "seeds": {"related": len(related), "close": len(close), "far": len(far)},
    "chosen": {"k": best_k, "weighted": best_w, "loo": loo[(best_k, best_w)]},
    "loo": [{"k": k, "weighted": w, **m} for (k, w), m in loo.items()],
    "holdout": {"n": len(hold_titles), "positives": int(sum(hold_y)), "knn": chosen, "baseline_single_cut": baseline,
                "baseline_cut": float(single_cut), "table": holdout_table},
    "ablations": ablations,
    "gray_zone": {"n": len(gray), "related": int((gray_scores >= 0.5).sum()),
                  "agreement": {f"{c}|{k}": n for (c, k), n in agree.items()}},
    "holdout_errors": errors,
    "variants": variants,
    "gray_zone_best_knn": {"related": int((bal_gray >= 0.5).sum()), "agreement": {f"{c}|{k}": n for (c, k), n in agree_bal.items()}},
}, indent=1, default=float))
print(f"wrote {RESULTS_PATH}")

wrote /Users/nivgreenstein/Desktop/projects/geoai/Corpus2GeoRAG_Pacific/Data/interim/knn_results.json
